# Jour 2 · Construire un RAG conversationnel


## Le problème nouveau

Le RAG du Jour 1 répondait à une question autonome. Dans un dialogue, l'utilisateur emploie rapidement des raccourcis : « Et si cela persiste ? », « Et sur un portable ? », « Pourquoi ? ».

Ces phrases sont compréhensibles avec les messages précédents, mais trop vagues pour une recherche sémantique isolée.


![Une question ambiguë devient une question autonome](../assets/jour_01/06_ambiguite_conversation.png)

*La reformulation récupère le sujet dans l'historique avant d'interroger la documentation.*


## Deux mémoires à ne pas confondre

Le mot **mémoire** peut induire en erreur :

- l'historique contient les derniers messages de cette conversation ;
- la base vectorielle contient les connaissances validées de l'entreprise.

L'historique aide à comprendre la question. Il ne devient pas une source documentaire fiable.


![Historique et documentation alimentent deux étapes différentes](../assets/jour_01/06_memoire_vs_connaissance.png)

*La mémoire explique le dialogue ; les documents fondent les faits de la réponse.*


In [ ]:
import sys
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# Les imports Python ne suivent pas automatiquement les chemins de fichiers.
# Cette ligne permet notamment d'importer le package local `apps` depuis un
# notebook lancé dans jour_03 ou corrections/jour_03.
root_text = str(ROOT)
if root_text not in sys.path:
    sys.path.insert(0, root_text)

print("Racine du cours :", ROOT)


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


In [ ]:
import hashlib

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def stable_chunk_id(chunk):
    raw = (
        f"{chunk.metadata.get('source', '')}|"
        f"{chunk.metadata.get('page', '')}|"
        f"{chunk.page_content}"
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def load_helpdesk_chunks():
    loaded = []
    for path in sorted(DOCUMENTS_DIR.glob("*.md")):
        if path.name == "README.md":
            continue
        docs = TextLoader(str(path), encoding="utf-8").load()
        for doc in docs:
            doc.metadata["source"] = path.name
        loaded.extend(docs)

    pdf_path = DOCUMENTS_DIR / "guide_escalade_p1.pdf"
    pdf_docs = PyPDFLoader(str(pdf_path)).load()
    for doc in pdf_docs:
        doc.metadata["source"] = pdf_path.name
    loaded.extend(pdf_docs)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=450,
        chunk_overlap=80,
        separators=["\n## ", "\n\n", "\n", " ", ""],
    )
    return splitter.split_documents(loaded)


database_dir = OUTPUTS_DIR / "chroma_helpdesk"
vectorstore = Chroma(
    collection_name="helpdesk_documents",
    embedding_function=embeddings,
    persist_directory=str(database_dir),
)

if len(vectorstore.get()["ids"]) == 0:
    helpdesk_chunks = load_helpdesk_chunks()
    vectorstore.add_documents(
        helpdesk_chunks,
        ids=[stable_chunk_id(chunk) for chunk in helpdesk_chunks],
    )

print("Chunks disponibles :", len(vectorstore.get()["ids"]))


In [ ]:
def format_docs(docs):
    return "\n\n".join(
        f"[Source : {doc.metadata.get('source', 'inconnue')}]\n{doc.page_content}"
        for doc in docs
    )


## Observer l'échec sans historique

Nous cherchons directement avec la question de suivi. Le retriever reçoit le mot « cela » sans savoir qu'il désigne une erreur VPN.


In [ ]:
question_ambigue = "Et si cela persiste ?"
docs_directs = vectorstore.similarity_search(question_ambigue, k=3)

for doc in docs_directs:
    print("-", doc.metadata.get("source"))


## Reformuler avant de rechercher

Une **question autonome** contient les informations nécessaires sans relire la conversation. Le LLM ne répond pas encore : il réécrit uniquement la requête.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


prompt_reformulation = ChatPromptTemplate.from_messages([
    (
        "system",
        "Réécris la dernière question afin qu'elle soit compréhensible sans l'historique. "
        "N'y réponds pas. Conserve le besoin exact et retourne une seule phrase.",
    ),
    (
        "human",
        "HISTORIQUE :\n{historique}\n\nDERNIÈRE QUESTION :\n{question}",
    ),
])
chaine_reformulation = prompt_reformulation | llm | StrOutputParser()


historique_texte = """Utilisateur : GlobalProtect affiche « portail inaccessible ».
Assistant : Je vais chercher la procédure VPN correspondante."""

question_autonome = chaine_reformulation.invoke({
    "historique": historique_texte,
    "question": question_ambigue,
})
print(question_autonome)


## Comparer les documents retrouvés

La reformulation n'est utile que si elle améliore la recherche. Nous affichons donc les sources au lieu de faire confiance à une phrase qui semble correcte.


In [ ]:
docs_reformules = vectorstore.similarity_search(question_autonome, k=3)
for doc in docs_reformules:
    print("-", doc.metadata.get("source"))
    print(" ", doc.page_content[:160].replace("\n", " "))


## Répondre avec les documents

Le prompt de réponse reçoit la question autonome et les passages retrouvés. Les sources continuent de venir des métadonnées, pas de la mémoire du modèle.


In [ ]:
prompt_reponse = ChatPromptTemplate.from_messages([
    (
        "system",
        "Réponds uniquement avec le contexte. Si l'information manque, réponds : "
        "Je ne trouve pas cette information dans la documentation fournie.\n\n"
        "CONTEXTE :\n{contexte}",
    ),
    ("human", "{question}"),
])
chaine_reponse = prompt_reponse | llm | StrOutputParser()

reponse = chaine_reponse.invoke({
    "contexte": format_docs(docs_reformules),
    "question": question_autonome,
})
sources = sorted({doc.metadata.get("source") for doc in docs_reformules})
print(reponse)
print("Sources :", sources)


## Représenter l'historique simplement

Pour apprendre, une liste de dictionnaires suffit. Chaque élément possède un rôle et un contenu. Une application réelle devra aussi séparer les conversations de différents utilisateurs.


In [ ]:
historique = [
    {"role": "user", "content": "GlobalProtect affiche portail inaccessible."},
    {"role": "assistant", "content": "Vérifiez d'abord votre connexion Internet."},
    {"role": "user", "content": "Et si cela persiste ?"},
]

for message in historique:
    print(message["role"], "→", message["content"])


### À vous de jouer — borner l'historique

Une liste qui grandit sans limite augmente le prompt, la latence et le risque de bruit. Conservez uniquement les derniers éléments.


In [ ]:
# À vous de jouer — retournez seulement les `maximum` derniers éléments.
def limiter_historique(messages, maximum=6):
    return messages


exemple = list(range(10))
print(limiter_historique(exemple, maximum=4))


## Ce que la mémoire ne résout pas

- Elle ne rend pas un document faux plus fiable.
- Elle ne garantit pas une bonne reformulation.
- Elle disparaît avec le noyau si elle reste seulement en mémoire vive.
- Elle doit être bornée, séparée par conversation et testée.

## À retenir

Reformuler d'abord, rechercher ensuite, répondre enfin. À chaque étape, la donnée a un rôle différent.
